In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

BASE = "/content/drive/MyDrive/ASD_XAI_Project/results"

combined_df = pd.read_csv(
    f"{BASE}/combined_xai_slice_ranking.csv"
)

print("Rows:", len(combined_df))
print(combined_df.columns.tolist())
print(combined_df.head())

Rows: 1054
['SUB_ID', 'SLICE_PATH', 'LABEL', 'SHAP_SCORE', 'PERMUTATION_SCORE', 'SHAP_NORM', 'PERMUTATION_NORM', 'COMBINED_SCORE', 'SLICE_RANK']
   SUB_ID                                         SLICE_PATH    LABEL  \
0   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
1   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
2   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
3   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
4   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   

   SHAP_SCORE  PERMUTATION_SCORE  SHAP_NORM  PERMUTATION_NORM  COMBINED_SCORE  \
0    0.000295           0.004262   0.769004          0.006120        0.387562   
1    0.000283           0.058933   0.721816          0.084635        0.403226   
2    0.000245           0.036996   0.578783          0.053131        0.315957   
3    0.000214           0.037339   0.460351          0.053623        0.256987   
4    0.0002

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

MODEL_PATH = (
    "/content/drive/MyDrive/ASD_XAI_Project/models/"
    "efficientnet_baseline.keras"
)

model = tf.keras.models.load_model(MODEL_PATH)

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image


def evaluate_subject_level(df, model):

    results = []

    for sub_id, group in df.groupby("SUB_ID"):

        images = np.stack([
            load_image(path).numpy()
            for path in group["SLICE_PATH"]
        ])

        predictions = model.predict(
            images,
            batch_size=16,
            verbose=0
        ).reshape(-1)

        subject_probability = np.mean(predictions)

        predicted_label = (
            "ASD"
            if subject_probability >= 0.5
            else "Control"
        )

        true_label = group["LABEL"].iloc[0]

        results.append({
            "SUB_ID": sub_id,
            "TRUE_LABEL": true_label,
            "PREDICTED_LABEL": predicted_label,
            "PROBABILITY": subject_probability,
            "NUM_SLICES": len(group)
        })

    results_df = pd.DataFrame(results)

    accuracy = accuracy_score(
        results_df["TRUE_LABEL"],
        results_df["PREDICTED_LABEL"]
    )

    return results_df, accuracy


print("Model loaded!")
print(model.input_shape)

Model loaded!
(None, 224, 224, 3)


In [4]:
WEIGHT_VALUES = [
    (0.2, 0.8),
    (0.3, 0.7),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

K_VALUES = [5, 10, 15, 20]

experiment_results = []

for alpha, beta in WEIGHT_VALUES:

    temp_df = combined_df.copy()

    temp_df["NEW_COMBINED_SCORE"] = (
        alpha * temp_df["SHAP_NORM"]
        + beta * temp_df["PERMUTATION_NORM"]
    )

    temp_df["NEW_RANK"] = (
        temp_df
        .groupby("SUB_ID")["NEW_COMBINED_SCORE"]
        .rank(
            method="first",
            ascending=False
        )
    )

    for k in K_VALUES:

        selected = temp_df[
            temp_df["NEW_RANK"] <= k
        ].copy()

        _, accuracy = evaluate_subject_level(
            selected,
            model
        )

        experiment_results.append({
            "SHAP_WEIGHT": alpha,
            "PERM_WEIGHT": beta,
            "TOP_K": k,
            "TOTAL_SLICES": len(selected),
            "ACCURACY": accuracy
        })

        print(
            f"SHAP={alpha:.1f} | "
            f"PERM={beta:.1f} | "
            f"K={k:2d} | "
            f"Accuracy={accuracy:.4f}"
        )


weight_results_df = pd.DataFrame(
    experiment_results
)

print("\n=== ALL RESULTS ===")
print(weight_results_df)

best = weight_results_df.loc[
    weight_results_df["ACCURACY"].idxmax()
]

print("\n=== BEST VALIDATION RESULT ===")
print(best)

SHAP=0.2 | PERM=0.8 | K= 5 | Accuracy=0.3333
SHAP=0.2 | PERM=0.8 | K=10 | Accuracy=0.3333
SHAP=0.2 | PERM=0.8 | K=15 | Accuracy=0.3333
SHAP=0.2 | PERM=0.8 | K=20 | Accuracy=0.3333
SHAP=0.3 | PERM=0.7 | K= 5 | Accuracy=0.3333
SHAP=0.3 | PERM=0.7 | K=10 | Accuracy=0.3333
SHAP=0.3 | PERM=0.7 | K=15 | Accuracy=0.3333
SHAP=0.3 | PERM=0.7 | K=20 | Accuracy=0.3333
SHAP=0.4 | PERM=0.6 | K= 5 | Accuracy=0.3333
SHAP=0.4 | PERM=0.6 | K=10 | Accuracy=0.3333
SHAP=0.4 | PERM=0.6 | K=15 | Accuracy=0.3333
SHAP=0.4 | PERM=0.6 | K=20 | Accuracy=0.3333
SHAP=0.5 | PERM=0.5 | K= 5 | Accuracy=0.3333
SHAP=0.5 | PERM=0.5 | K=10 | Accuracy=0.3333
SHAP=0.5 | PERM=0.5 | K=15 | Accuracy=0.3333
SHAP=0.5 | PERM=0.5 | K=20 | Accuracy=0.3333
SHAP=0.6 | PERM=0.4 | K= 5 | Accuracy=0.3333
SHAP=0.6 | PERM=0.4 | K=10 | Accuracy=0.3333
SHAP=0.6 | PERM=0.4 | K=15 | Accuracy=0.3333
SHAP=0.6 | PERM=0.4 | K=20 | Accuracy=0.3333
SHAP=0.7 | PERM=0.3 | K= 5 | Accuracy=0.3333
SHAP=0.7 | PERM=0.3 | K=10 | Accuracy=0.3333
SHAP=0.7 |

In [5]:
BEST_SHAP_WEIGHT = 0.8
BEST_PERM_WEIGHT = 0.2
BEST_K = 5

best_df = combined_df.copy()

best_df["BEST_SCORE"] = (
    BEST_SHAP_WEIGHT * best_df["SHAP_NORM"]
    + BEST_PERM_WEIGHT * best_df["PERMUTATION_NORM"]
)

best_df["BEST_RANK"] = (
    best_df
    .groupby("SUB_ID")["BEST_SCORE"]
    .rank(method="first", ascending=False)
)

best_selected_df = (
    best_df[
        best_df["BEST_RANK"] <= BEST_K
    ]
    .copy()
)

best_results, best_accuracy = evaluate_subject_level(
    best_selected_df,
    model
)

print("=== BEST CONFIGURATION ===")
print("SHAP weight:", BEST_SHAP_WEIGHT)
print("Permutation weight:", BEST_PERM_WEIGHT)
print("Top-K:", BEST_K)

print("\nSubject results:")
print(best_results)

print("\nAccuracy:", best_accuracy)

=== BEST CONFIGURATION ===
SHAP weight: 0.8
Permutation weight: 0.2
Top-K: 5

Subject results:
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.292052           5
1   51463        ASD             ASD     0.628599           5
2   51466        ASD             ASD     0.562951           5
3   51475    Control             ASD     0.724719           5
4   51476    Control         Control     0.439795           5
5   51487    Control             ASD     0.582153           5

Accuracy: 0.5


In [6]:
import os

EXP2_DIR = (
    "/content/drive/MyDrive/ASD_XAI_Project/"
    "experiment_02_weight_optimization"
)

os.makedirs(EXP2_DIR, exist_ok=True)

# Save all weight experiments
weight_results_df.to_csv(
    f"{EXP2_DIR}/weight_optimization_results.csv",
    index=False
)

# Save selected Top-5 slices
best_selected_df.to_csv(
    f"{EXP2_DIR}/best_selected_slices.csv",
    index=False
)

# Save subject-level predictions
best_results.to_csv(
    f"{EXP2_DIR}/best_subject_results.csv",
    index=False
)

print("Experiment 02 saved!")
print("Folder:", EXP2_DIR)
print("Best accuracy:", best_accuracy)

Experiment 02 saved!
Folder: /content/drive/MyDrive/ASD_XAI_Project/experiment_02_weight_optimization
Best accuracy: 0.5
